<a href="https://colab.research.google.com/github/vthkrl/big_data_fall26/blob/main/Big_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


See `blockify_years.ipynb` for various cleaning tasks - notably, converting yearly data into five-year blocks for the crime rate, fine particles, median income, and premature mortality metrics. This notebook operates on the cleaned CSVs produced by that one.

All the metric tables consumed by this notebook should have the following columns:

- `TimePeriod`, holding the five-year block (in the format `YYYY-YY`, ie `2015-19`)
- `GeoID`, holding the community district ID number
- Some other column with the actual metric value for that time period and community district

The name of the metric-value column should be specified in the `metric_colnames` table, along with the name to be used for that metric in the merged table.

<br>

**TODO: does the facilities table need cleanup?**



2. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




Load and combine data from cleaned CSVs

In [11]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/Project/datasets/"

spark = SparkSession.builder \
    .appName("FacilityPredictor") \
    .getOrCreate()

facilities_path = folder_path + "facilities.csv"
facilities = spark.read.csv(facilities_path, header=True, inferSchema=True)

# metrics
# 1. rent burden
rent_burden_rate_path = folder_path + "rent_burden.csv"
rent_burden_rate = spark.read.csv(rent_burden_rate_path, header=True, inferSchema=True)

# 2. crowding
crowding_rate_path = folder_path + "crowding.csv"
crowding_rate = spark.read.csv(crowding_rate_path, header=True, inferSchema=True)

# 3. median income
median_income_path = folder_path + "median_income_parsed.csv"
median_income = spark.read.csv(median_income_path, header=True, inferSchema=True)

# 4. poverty
poverty_rate_path = folder_path + "poverty.csv"
poverty_rate = spark.read.csv(poverty_rate_path, header=True, inferSchema=True)

# 5. crime
crime_rate_path = folder_path + "crime_rate_parsed.csv"
crime_rate = spark.read.csv(crime_rate_path, header=True, inferSchema=True)

# 6. high school graduation
high_school_graduation_rate_path = folder_path + "high_school.csv"
high_school_graduation_rate = spark.read.csv(high_school_graduation_rate_path, header=True, inferSchema=True)

# 7. fine particles
fine_particles_concentration_path = folder_path + "fine_particles_parsed.csv"
fine_particles_concentration = spark.read.csv(fine_particles_concentration_path, header=True, inferSchema=True)

# 8. walking to work rate
walk_to_work_rate = folder_path + "walk_to_work.csv"
walk_to_work_rate = spark.read.csv(walk_to_work_rate, header=True, inferSchema=True)

# 9. infant mortality rate
infant_mortality_rate_path = folder_path + "infant_mortality_parsed.csv"
infant_mortality_rate = spark.read.csv(infant_mortality_rate_path, header=True, inferSchema=True)

# 10. independent living difficulty
indep_difficulty_rate_path = folder_path + "indep_living.csv"
indep_difficulty_rate = spark.read.csv(indep_difficulty_rate_path, header=True, inferSchema=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
metric_colnames = [
    (rent_burden_rate, "Percent", "rent_burden_pct"),
    (crowding_rate, "Percent", "crowding_pct"),
    (median_income, "median_income", "median_income"),
    (poverty_rate, "Percent", "poverty_pct"),
    (crime_rate, "crime_all_rt", "crimes_per_1000_ppl"),
    (high_school_graduation_rate, "Percent", "high_school_grad_pct"),
    (fine_particles_concentration, "Annual mean mcg/m3", "fine_particle_concentration"),
    (walk_to_work_rate, "Walked (percent)", "walk_to_work_rate"),
    (infant_mortality_rate, "infant_mortality", "infant_deaths_per_1000_births"),
    (indep_difficulty_rate, "Percent", "indep_difficulty_pct")
]

# join all the individual metric tables by TimePeriod and GeoID, using left join to handle time periods absent from some datasets
mega_join = rent_burden_rate
select_terms = ["TimePeriod", "GeoID"]
for group in metric_colnames:
  if group[0] != rent_burden_rate:
    mega_join = mega_join.join(group[0], ["TimePeriod", "GeoID"], how="left")
  # create the appropriate term for the select statement, ie 'crowding_rate["Percent"].alias("crowding_pct")'
  select_terms.append(group[0][group[1]].alias(group[2]))

# select only the columns we actually care about
all_metrics = mega_join.select(*select_terms)
all_metrics.show(10)

+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+-----------------------------+--------------------+
|TimePeriod|GeoID|rent_burden_pct|crowding_pct|median_income|poverty_pct|crimes_per_1000_ppl|high_school_grad_pct|fine_particle_concentration|walk_to_work_rate|infant_deaths_per_1000_births|indep_difficulty_pct|
+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+-----------------------------+--------------------+
|   2015-19|  201|           58.4|        12.7|      30722.2|       40.1|              24.98|                63.8|                       8.54|              9.7|                         5.34|                 8.5|
|   2015-19|  202|           58.4|        12.7|      30722.2|       40.1|              19.32|                63.8|                       8.54|          

In [13]:
# run this cell to drop rows with any NULL values (optional)
all_metrics = all_metrics.dropna()

Standardize metric scores

In [41]:
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml import Pipeline

colNames = [group[2] for group in metric_colnames]

vecAssemble = VectorAssembler(inputCols=colNames, outputCol='metricVec')
scaler = StandardScaler(inputCol='metricVec', outputCol='scaledMetricVec')
preprocessPipeline = Pipeline(stages=[vecAssemble, scaler])

preModel = preprocessPipeline.fit(all_metrics)

scaled = preModel.transform(all_metrics) \
        .withColumn('unpack', vector_to_array('scaledMetricVec')) \
        .select(['GeoId', 'TimePeriod'] + [F.col('unpack')[i] for i in range(len(colNames))]) \
        .withColumnsRenamed({f'unpack[{i}]': colNames[i]+'_scaled' for i in range(len(colNames))})

scaled.show()

+-----+----------+----------------------+-------------------+--------------------+------------------+--------------------------+---------------------------+----------------------------------+------------------------+------------------------------------+---------------------------+
|GeoId|TimePeriod|rent_burden_pct_scaled|crowding_pct_scaled|median_income_scaled|poverty_pct_scaled|crimes_per_1000_ppl_scaled|high_school_grad_pct_scaled|fine_particle_concentration_scaled|walk_to_work_rate_scaled|infant_deaths_per_1000_births_scaled|indep_difficulty_pct_scaled|
+-----+----------+----------------------+-------------------+--------------------+------------------+--------------------------+---------------------------+----------------------------------+------------------------+------------------------------------+---------------------------+
|  201|   2015-19|     8.407487722505001| 2.6960523404697483|  0.9294841181361388| 3.955625173253574|         2.110677489484847|           6.3609967266950

Calculate overall quality metric

In [42]:
weights = [ # weights for:
    -1,   # rent_burden_pct
    -1,   # crowding_pct
    5,    # median_income
    -2,   # poverty_pct
    -1,   # crimes_per_1000_ppl
    4,    # high_school_grad_pct
    -0.5, # fine_particle_concentration
    3,    # walk_to_work_rate
    -1.5, # infant_deaths_per_1000_births
    -1    # indep_difficulty_pct
]
total_magnitude = sum(abs(w) for w in weights)

norm_weights = [w / total_magnitude for w in weights]

# OverallQuality ranges from 0.0 to 1.0
quality_score = F.lit(0.0)
for i, w in enumerate(norm_weights):
    column = F.col(colNames[i] + "_scaled")

    norm_metric = (1.0 - column) if w < 0 else column
    quality_score = quality_score + (abs(w) * norm_metric)

overall_quality = scaled.withColumn("OverallQuality", quality_score)

# comment out this line if you want to keep the scores and percentiles
overall_quality = overall_quality.select("GeoID", "TimePeriod", "OverallQuality")


top_10_cds = overall_quality.orderBy(F.col("OverallQuality").desc()).limit(10)
top_10_cds.show()

+-----+----------+------------------+
|GeoID|TimePeriod|    OverallQuality|
+-----+----------+------------------+
|  102|   2015-19| 3.382286172898492|
|  101|   2015-19| 3.368092875275037|
|  106|   2015-19| 3.355570092188716|
|  102|   2014-18| 3.291448302683907|
|  101|   2014-18|3.2773242079635843|
|  106|   2014-18| 3.259053240497676|
|  102|   2013-17|3.2371365768823885|
|  101|   2013-17|3.2329678509387243|
|  106|   2013-17|3.2139735090679444|
|  101|   2012-16|3.1934843993176756|
+-----+----------+------------------+



Add facility counts columns to make training table

In [43]:
# facilities
facility_counts = (
    facilities
    .groupBy("cd")
    .pivot("facgroup")
    .count()
    .na.fill(0)
).withColumnRenamed("cd", "GeoID")

trainer = overall_quality.join(
    facility_counts,
    on="GeoID",
    how="left"
)

trainer.show(10)

+-----+----------+-------------------+--------------+-----+--------------------------+---------------------------------------------+---------------------+-----------------------------+------------------+-----------+----------------+----------------+--------------+-----------------------+---------+-----------------------------+------------------------------+--------------+----------------+-------------+--------------+-----------+------------------+--------------+----------------------------------+--------------------+--------------+
|GeoID|TimePeriod|     OverallQuality|ADULT SERVICES|CAMPS|CHILD SERVICES AND WELFARE|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|CULTURAL INSTITUTIONS|DAY CARE AND PRE-KINDERGARTEN|EMERGENCY SERVICES|HEALTH CARE|HIGHER EDUCATION|HISTORICAL SITES|HUMAN SERVICES|JUSTICE AND CORRECTIONS|LIBRARIES|MATERIAL SUPPLIES AND MARKETS|OFFICES, TRAINING, AND TESTING|OTHER PROPERTY|PARKS AND PLAZAS|PUBLIC SAFETY|SCHOOLS (K-12)|SOLID WASTE|TELECOMMUNICATIONS|TRANSPORTAT

In [44]:
trainer.toPandas().to_csv(folder_path + "trainer.csv", index=False)

3. Model training: train a machine learning model on the raw scores for the quality of life domains of each district, so that the model determines how the count of a facility type in a district may affect the raw scores of domains for that district.


4. Model execution: for each (district, facility type) pair, use the model to calculate the predicted domain scores, then calculate the predicted domain percentiles and the Predicted Overall Percentile Lift.


In [ ]:
import re

cds = [row[0] for row in facilities.select("cd").distinct().collect()]
facs = [row[0] for row in facilities.select("facgroup").distinct().collect()]

resultArray = []
for cd in cds:
  counts = facility_counts.filter(f"cd = {cd}").collect()[0]
  # TODO: pass unmodified data into model, predict base score
  baseScore = 0.0
  newRowData = [cd]
  #print(f"processing cd #{cd}")
  for factype in facs:
    modified = counts.asDict()
    modified[factype] += 1
    # TODO: pass data into model, get a predicted score, subtract base score
    predictedLift = 0.0 - baseScore
    newRowData.append(predictedLift)
  resultArray.append(newRowData)

outSchema = "cd int"
for factype in facs:
  outSchema += f", `{factype}` double"
results = spark.createDataFrame(resultArray, outSchema)

results.show()

+---+---------+------------------------------+----------------+------------------+-----------------------------+-----+------------------+-----------------------+--------------+-----------+--------------+--------------+----------------------------------+---------------------+--------------+-----------+--------------------+--------------+----------------+-------------+--------------------------+-----------------------------+---------------------------------------------+----------------+--------------+
| cd|LIBRARIES|OFFICES, TRAINING, AND TESTING|HISTORICAL SITES|EMERGENCY SERVICES|MATERIAL SUPPLIES AND MARKETS|CAMPS|TELECOMMUNICATIONS|JUSTICE AND CORRECTIONS|TRANSPORTATION|SOLID WASTE|HUMAN SERVICES|OTHER PROPERTY|VOCATIONAL AND PROPRIETARY SCHOOLS|CULTURAL INSTITUTIONS|SCHOOLS (K-12)|HEALTH CARE|WATER AND WASTEWATER|YOUTH SERVICES|PARKS AND PLAZAS|PUBLIC SAFETY|CHILD SERVICES AND WELFARE|DAY CARE AND PRE-KINDERGARTEN|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|HIGHER EDUCATION|ADULT 

5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.